In [0]:
# Configure Spark to authenticate to ADLS using the storage key from Key Vault
storage_account_name = "aparnastshellenergyuks"
storage_account_key = dbutils.secrets.get(
    scope="aparna-kv-shellenergy-scope",
    key="aparna-adls-storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Define base paths for each medallion layer
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/batch"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/batch"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/batch"

print("ADLS authentication configured.")
print(f"Bronze: {bronze_path}")
print(f"Silver: {silver_path}")
print(f"Gold: {gold_path}")

In [0]:
# Confirm we can see the files ADF copied into Bronze
display(dbutils.fs.ls(bronze_path))

In [0]:
# Customers: Bronze → Silver
from pyspark.sql.functions import col, to_date, trim, current_timestamp
from pyspark.sql.types import StringType

# Read raw customers CSV from Bronze
df_customers_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{bronze_path}/customers/")
)

print(f"Bronze customers row count: {df_customers_bronze.count()}")
df_customers_bronze.printSchema()
display(df_customers_bronze.limit(5))

In [0]:
# Inspect the companies_house_no issue, then clean and write to Silver
# Quick check: did we lose leading zeros on companies_house_no?
display(df_customers_bronze.select("customer_id", "companies_house_no").limit(5))

# Clean + type correctly, then write as Delta to Silver
df_customers_silver = (
    df_customers_bronze
    .withColumn("companies_house_no", col("companies_house_no").cast(StringType()))
    .withColumn("companies_house_no", 
                # Pad back to 8 digits in case leading zeros were lost during CSV inference
                col("companies_house_no").cast(StringType()))
    .withColumn("company_name", trim(col("company_name")))
    .withColumn("account_manager", trim(col("account_manager")))
    .withColumn("_silver_ingested_at", current_timestamp())
    .dropDuplicates(["customer_id"])  # defensive dedup, in case of any upstream duplication
)

# Re-pad companies_house_no to 8 digits using Spark's lpad function
from pyspark.sql.functions import lpad
df_customers_silver = df_customers_silver.withColumn(
    "companies_house_no", lpad(col("companies_house_no"), 8, "0")
)

display(df_customers_silver.limit(5))
print(f"Silver customers row count: {df_customers_silver.count()}")

In [0]:
# Write as a managed Delta table in Silver
df_customers_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/customers/")

# Optional: also register it as a queryable table in the metastore for easier SQL access later
'''spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver_customers
    USING DELTA
    LOCATION '{silver_path}/customers/'
""")'''

print("customers written to Silver as Delta table.")

In [0]:
# Confirm we can read it back directly by path (this is how we'll read Silver/Gold tables 
# throughout the rest of the project - by path, not by metastore table name)
df_check = spark.read.format("delta").load(f"{silver_path}/customers/")
display(df_check.limit(5))
print(f"Row count: {df_check.count()}")

In [0]:
#Sites: Bronze → Silver
# Read raw sites CSV from Bronze
df_sites_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{bronze_path}/sites/")
)

print(f"Bronze sites row count: {df_sites_bronze.count()}")
df_sites_bronze.printSchema()
display(df_sites_bronze.limit(5))

In [0]:
from pyspark.sql.functions import col, trim, current_timestamp, lpad

# Clean + correctly type, padding mpan back to 13 digits (UK MPAN standard length)
df_sites_silver = (
    df_sites_bronze
    .withColumn("mpan", lpad(col("mpan").cast("string"), 13, "0"))
    .withColumn("site_name", trim(col("site_name")))
    .withColumn("postcode", trim(col("postcode")))
    .withColumn("_silver_ingested_at", current_timestamp())
    .dropDuplicates(["site_id"])
)

display(df_sites_silver.select("site_id", "mpan", "postcode").limit(10))
print(f"Silver sites row count: {df_sites_silver.count()}")

In [0]:
# Write as Delta to Silver
df_sites_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/sites/")

# Verify by reading back
df_check = spark.read.format("delta").load(f"{silver_path}/sites/")
print(f"Verified row count: {df_check.count()}")

In [0]:
df_contracts_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{bronze_path}/contracts/")
)

print(f"Bronze contracts row count: {df_contracts_bronze.count()}")
df_contracts_bronze.printSchema()
display(df_contracts_bronze.limit(10))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col

# For each site, order contracts by start_date and check if any contract's start_date
# falls before the PREVIOUS contract's end_date for that same site (a true overlap)
window_spec = Window.partitionBy("site_id").orderBy("start_date")

df_overlap_check = (
    df_contracts_bronze
    .withColumn("next_start_date", lead("start_date").over(window_spec))
    .withColumn(
        "has_overlap",
        col("next_start_date").isNotNull() & (col("next_start_date") <= col("end_date"))
    )
)

overlap_count = df_overlap_check.filter(col("has_overlap") == True).count()
print(f"Contracts with overlapping date ranges: {overlap_count}")

if overlap_count > 0:
    display(df_overlap_check.filter(col("has_overlap") == True).select(
        "site_id", "contract_id", "start_date", "end_date", "next_start_date"
    ))

In [0]:
from pyspark.sql.functions import current_timestamp

df_contracts_silver = (
    df_contracts_bronze
    .withColumn("_silver_ingested_at", current_timestamp())
    .dropDuplicates(["contract_id"])
)

print(f"Silver contracts row count: {df_contracts_silver.count()}")

df_contracts_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/contracts/")

# Verify
df_check = spark.read.format("delta").load(f"{silver_path}/contracts/")
print(f"Verified row count: {df_check.count()}")

In [0]:
df_prices_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{bronze_path}/wholesale_prices/")
)

print(f"Bronze wholesale_prices row count: {df_prices_bronze.count()}")
df_prices_bronze.printSchema()
display(df_prices_bronze.limit(5))

In [0]:
from pyspark.sql.functions import current_timestamp

df_prices_silver = (
    df_prices_bronze
    .withColumn("_silver_ingested_at", current_timestamp())
    .dropDuplicates(["price_id"])
)

print(f"Silver wholesale_prices row count: {df_prices_silver.count()}")

df_prices_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/wholesale_prices/")

# Verify
df_check = spark.read.format("delta").load(f"{silver_path}/wholesale_prices/")
print(f"Verified row count: {df_check.count()}")

In [0]:
df_readings_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{bronze_path}/meter_readings/")
)

print(f"Bronze meter_readings row count: {df_readings_bronze.count()}")
df_readings_bronze.printSchema()
display(df_readings_bronze.limit(10))

# Check the status/flag distribution
display(df_readings_bronze.groupBy("meter_status", "data_quality_flag").count().orderBy("count", ascending=False))

In [0]:
from pyspark.sql.functions import explode, sequence, to_timestamp, lit, expr, col
from pyspark.sql.functions import min as spark_min, max as spark_max, last
from pyspark.sql.window import Window

date_bounds = df_readings_bronze.select(
    spark_min("interval_start").alias("min_ts"),
    spark_max("interval_start").alias("max_ts")
).collect()[0]
min_ts, max_ts = date_bounds["min_ts"], date_bounds["max_ts"]
print(f"Data range: {min_ts} to {max_ts}")

site_ids_df = df_readings_bronze.select("site_id").distinct()
print(f"Distinct sites: {site_ids_df.count()}")

# IMPORTANT: sequence() takes the step as its 3rd POSITIONAL argument, not a keyword.
# This is the line that was wrong before - confirmed correct syntax below.
calendar_df = spark.createDataFrame([(1,)], ["dummy"]).select(
    explode(
        sequence(lit(min_ts), lit(max_ts), expr("interval 30 minutes"))
    ).alias("interval_start")
)

full_grid = site_ids_df.crossJoin(calendar_df)
print(f"Expected full grid row count: {full_grid.count()}")

df_with_gaps = full_grid.join(
    df_readings_bronze, on=["site_id", "interval_start"], how="left"
)

missing_count = df_with_gaps.filter(col("kwh_interval").isNull()).count()
print(f"Missing intervals detected: {missing_count} "
      f"({100 * missing_count / full_grid.count():.3f}% of expected grid)")

In [0]:
from pyspark.sql.functions import last, expr, current_timestamp

# Forward-fill kwh_interval using the last known value per site, ordered by time.
# rowsBetween(unboundedPreceding, 0) means "look backwards from this row to the 
# start of this site's data" - ignorenulls=True skips over the gaps themselves
window_spec = Window.partitionBy("site_id").orderBy("interval_start").rowsBetween(
    Window.unboundedPreceding, 0
)

df_readings_filled = (
    df_with_gaps
    .withColumn("kwh_interval_filled", last("kwh_interval", ignorenulls=True).over(window_spec))
    .withColumn(
        "meter_status_final",
        expr("CASE WHEN kwh_interval IS NULL THEN 'Estimated' ELSE meter_status END")
    )
    .withColumn(
        "data_quality_flag_final",
        expr("CASE WHEN kwh_interval IS NULL THEN 'GAP_FILLED' ELSE data_quality_flag END")
    )
)

# Drop the original (pre-fill) columns and rename the final ones to clean names
df_readings_silver = (
    df_readings_filled
    .drop("kwh_interval", "meter_status", "data_quality_flag", "reading_id")
    .withColumnRenamed("kwh_interval_filled", "kwh_interval")
    .withColumnRenamed("meter_status_final", "meter_status")
    .withColumnRenamed("data_quality_flag_final", "data_quality_flag")
    .withColumn("_silver_ingested_at", current_timestamp())
)

# Sanity checks before writing
print(f"Silver meter_readings row count: {df_readings_silver.count()}")
print(f"Rows still null after fill (should be 0, would mean a site's very first "
      f"interval was itself missing): {df_readings_silver.filter(col('kwh_interval').isNull()).count()}")

display(df_readings_silver.groupBy("data_quality_flag").count().orderBy("count", ascending=False))

In [0]:
df_readings_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/meter_readings/")

# Verify
df_check = spark.read.format("delta").load(f"{silver_path}/meter_readings/")
print(f"Verified row count: {df_check.count()}")
display(df_check.groupBy("data_quality_flag").count().orderBy("count", ascending=False))

In [0]:
df_invoices_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{bronze_path}/billing_invoices/")
)

print(f"Bronze billing_invoices row count: {df_invoices_bronze.count()}")
df_invoices_bronze.printSchema()
display(df_invoices_bronze.limit(5))

In [0]:
from pyspark.sql.functions import current_timestamp

df_invoices_silver = (
    df_invoices_bronze
    .withColumn("_silver_ingested_at", current_timestamp())
    .dropDuplicates(["invoice_id"])
)

print(f"Silver billing_invoices row count: {df_invoices_silver.count()}")

df_invoices_silver.write.format("delta").mode("overwrite").save(f"{silver_path}/billing_invoices/")

# Verify
df_check = spark.read.format("delta").load(f"{silver_path}/billing_invoices/")
print(f"Verified row count: {df_check.count()}")